# Analysis notebook (load saved models)

This notebook does **analysis only** using models saved in Drive by `hourly.ipynb` and `15min.ipynb`.

Outputs are saved to:
- `/content/drive/MyDrive/Shared-Colab-Storage/Final/hourly/analysis/`
- `/content/drive/MyDrive/Shared-Colab-Storage/Final/15min/analysis/`


## 0) Setup


In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn tensorflow shap lime scipy

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import tensorflow as tf

from lime.lime_tabular import LimeTabularExplainer
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from tensorflow.keras import backend as K
from tensorflow.keras.models import load_model


def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_true - y_pred)))


## Runtime controls (can reduce for quick run)


In [ ]:
TRACKS = ["hourly", "15min"]
STACKS = ["single", "double", "bidir"]
WINDOWS = {
    "hourly": [1, 4, 8, 12, 16, 24, 36, 48, 74, 168, 336, 672],
    "15min": [1, 4, 8, 16, 24, 48, 64, 96, 672],
}

# Heavy analysis controls
MAX_MODELS_FOR_XAI = None     # None = all, or int like 6
USE_BEST_PLUS_STANDARD = True # if True, analyze [best] + XAI_WINDOWS_BASE
XAI_WINDOWS_BASE = [8, 24, 168]

SHAP_BG = 50
SHAP_TEST = 15
LIME_TEST = 10
LIME_SAMPLES = 200
IG_SAMPLES = 20
IG_STEPS = 50

MEMORY_ERASURE_CUTOFFS = [0.0, 0.25, 0.5, 0.75]
FIDELITY_TOP_K = 2

DRIVE_ROOT = Path('/content/drive/MyDrive/Shared-Colab-Storage/Final')
print('Drive root:', DRIVE_ROOT)


In [ ]:
def wia(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    m = np.mean(y_true)
    den = np.sum((np.abs(y_pred - m) + np.abs(y_true - m)) ** 2)
    if den == 0:
        return np.nan
    return 1 - np.sum((y_pred - y_true) ** 2) / den


def compute_metrics(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'wia': float(wia(y_true, y_pred)),
    }


def to_kwh(vals, scaler, target_idx, n_features):
    vals = np.asarray(vals).ravel()
    d = np.zeros((len(vals), n_features))
    d[:, target_idx] = vals
    return scaler.inverse_transform(d)[:, target_idx]


def load_track_assets(track):
    base = DRIVE_ROOT / track
    data_csv = base / 'data.csv'
    scaler_pkl = base / 'scaler.pkl'
    metrics_csv = base / 'results_metrics.csv'

    missing = [str(p) for p in [data_csv, scaler_pkl] if not p.exists()]
    if missing:
        print(f"[{track}] missing required files:", missing)
        return None

    scaled_df = pd.read_csv(data_csv)
    scaler = pickle.load(open(scaler_pkl, 'rb'))
    feature_cols = list(scaled_df.columns)
    target_idx = 0

    metrics_df = pd.read_csv(metrics_csv) if metrics_csv.exists() else None

    return {
        'track': track,
        'base': base,
        'scaled_df': scaled_df,
        'scaler': scaler,
        'feature_cols': feature_cols,
        'target_idx': target_idx,
        'metrics_df': metrics_df,
    }


def build_window_data(scaled_df, feature_cols, window, test_ratio=0.18):
    arr = scaled_df[feature_cols].to_numpy(dtype=np.float32)
    X, y = [], []
    for i in range(len(arr) - window):
        X.append(arr[i:i+window])
        y.append(arr[i+window, 0])
    X = np.array(X)
    y = np.array(y)
    split = int(len(X) * (1 - test_ratio))
    return X[:split], X[split:], y[:split], y[split:]


def load_saved_model(base, stack, window):
    model_path = base / stack / 'models' / f'win{window}.keras'
    if not model_path.exists():
        return None, model_path
    model = load_model(str(model_path), custom_objects={'rmse': rmse})
    return model, model_path


def get_best_from_metrics(metrics_df):
    if metrics_df is None or len(metrics_df) == 0:
        return None
    df = metrics_df.copy()
    col = 'rmse_kwh' if 'rmse_kwh' in df.columns else 'rmse'
    best = df.loc[df[col].idxmin()]
    return {'model': str(best['model']), 'window': int(best['window']), 'rmse_col': col, 'rmse': float(best[col])}


def integrated_gradients(model, x, baseline, steps=50):
    x_t = tf.convert_to_tensor(x[np.newaxis, ...], dtype=tf.float32)
    b_t = tf.convert_to_tensor(baseline[np.newaxis, ...], dtype=tf.float32)
    grads_sum = tf.zeros_like(x_t)

    for alpha in tf.linspace(0.0, 1.0, steps + 1):
        with tf.GradientTape() as tape:
            interp = b_t + alpha * (x_t - b_t)
            tape.watch(interp)
            pred = model(interp, training=False)
        grads_sum += tape.gradient(pred, interp)

    attr = ((x_t - b_t) * (grads_sum / float(steps + 1))).numpy()[0]
    return attr


## 1) Helpers


## 2) Load track assets + find best model per track


In [ ]:
track_assets = {}
best_models = {}

for track in TRACKS:
    assets = load_track_assets(track)
    if assets is None:
        continue
    track_assets[track] = assets
    best = get_best_from_metrics(assets['metrics_df'])
    if best:
        best_models[track] = best
        print(f"[{track}] best ->", best)
    else:
        print(f"[{track}] metrics missing, best model unavailable")

for track, assets in track_assets.items():
    analysis_dir = assets['base'] / 'analysis'
    analysis_dir.mkdir(parents=True, exist_ok=True)

( DRIVE_ROOT / 'analysis').mkdir(parents=True, exist_ok=True)
with open(DRIVE_ROOT / 'analysis' / 'best_models.json', 'w') as f:
    json.dump(best_models, f, indent=2)

print('Saved:', DRIVE_ROOT / 'analysis' / 'best_models.json')


## 3) Recompute metrics for all saved models


In [ ]:
for track, assets in track_assets.items():
    base = assets['base']
    scaler = assets['scaler']
    scaled_df = assets['scaled_df']
    feature_cols = assets['feature_cols']
    n_features = len(feature_cols)

    rows = []
    for stack in STACKS:
        for window in WINDOWS[track]:
            model, path = load_saved_model(base, stack, window)
            if model is None:
                continue

            X_train, X_test, y_train, y_test = build_window_data(scaled_df, feature_cols, window)
            y_pred = model.predict(X_test, verbose=0).ravel()

            yt = to_kwh(y_test, scaler, 0, n_features)
            yp = to_kwh(y_pred, scaler, 0, n_features)

            m = compute_metrics(yt, yp)
            rows.append({
                'track': track,
                'model': stack,
                'window': window,
                'rmse_kwh': m['rmse'],
                'mae_kwh': m['mae'],
                'r2': m['r2'],
                'wia': m['wia'],
                'model_path': str(path),
            })

    df = pd.DataFrame(rows)
    out = base / 'analysis' / 'recomputed_metrics.csv'
    if len(df) > 0:
        df.to_csv(out, index=False)
        print(f"[{track}] wrote", out, "rows:", len(df))
        display(df.sort_values('rmse_kwh').head(10))
    else:
        print(f"[{track}] no saved models found")


## 4) Build model list for heavy XAI (runtime controlled)


In [ ]:
models_for_xai = []

for track, assets in track_assets.items():
    best = best_models.get(track)
    candidates = []

    if USE_BEST_PLUS_STANDARD and best:
        win_set = set([best['window']] + [w for w in XAI_WINDOWS_BASE if w in WINDOWS[track]])
    else:
        win_set = set(WINDOWS[track])

    for stack in STACKS:
        for window in sorted(win_set):
            model, model_path = load_saved_model(assets['base'], stack, window)
            if model is not None:
                candidates.append((track, stack, window, model_path))

    if MAX_MODELS_FOR_XAI is not None:
        candidates = candidates[:MAX_MODELS_FOR_XAI]

    models_for_xai.extend(candidates)

print('Models selected for heavy XAI:', len(models_for_xai))
for item in models_for_xai[:20]:
    print(item)


## 5) SHAP + LIME + agreement (Spearman)


In [ ]:
for track, stack, window, model_path in models_for_xai:
    assets = track_assets[track]
    base = assets['base']
    scaled_df = assets['scaled_df']
    feature_cols = assets['feature_cols']

    xai_dir = base / 'analysis' / 'xai' / stack
    xai_dir.mkdir(parents=True, exist_ok=True)

    shap_csv = xai_dir / f'win{window}_shap.csv'
    lime_csv = xai_dir / f'win{window}_lime.csv'
    corr_csv = xai_dir / f'win{window}_shap_lime_corr.csv'

    model = load_model(str(model_path), custom_objects={'rmse': rmse})
    X_train, X_test, y_train, y_test = build_window_data(scaled_df, feature_cols, window)

    n_feat = X_train.shape[2]
    labels = [f"t-{window-1-t}_{feature_cols[f]}" for t in range(window) for f in range(n_feat)]

    if not shap_csv.exists():
        bg_n = min(SHAP_BG, len(X_train))
        test_n = min(SHAP_TEST, len(X_test))
        bg = X_train[:bg_n].reshape(bg_n, window * n_feat)
        test_flat = X_test[:test_n].reshape(test_n, window * n_feat)

        explainer = shap.KernelExplainer(
            lambda x: model.predict(x.reshape(-1, window, n_feat), verbose=0).ravel(), bg
        )
        sv = explainer.shap_values(test_flat, nsamples=40)

        shap_df = pd.DataFrame({'feature': labels, 'value': np.abs(sv).mean(axis=0)})
        shap_df['attr'] = shap_df['feature'].str.replace(r'^t-\d+_', '', regex=True)
        shap_rank = shap_df.groupby('attr', as_index=False)['value'].sum().sort_values('value', ascending=False)
        shap_rank.to_csv(shap_csv, index=False)

    if not lime_csv.exists():
        train_flat = X_train.reshape(len(X_train), window * n_feat)
        test_n = min(LIME_TEST, len(X_test))
        test_flat = X_test[:test_n].reshape(test_n, window * n_feat)

        lime_exp = LimeTabularExplainer(train_flat, mode='regression', verbose=False)
        importance = np.zeros(n_feat)

        for i in range(test_n):
            exp = lime_exp.explain_instance(
                test_flat[i],
                lambda x: model.predict(x.reshape(-1, window, n_feat), verbose=0).ravel(),
                num_samples=LIME_SAMPLES,
                num_features=window * n_feat,
            )
            for feat, wt in exp.as_list():
                for j, nm in enumerate(feature_cols):
                    if nm in feat:
                        importance[j] += abs(wt)

        lime_df = pd.DataFrame({'attr': feature_cols, 'value': importance / max(test_n, 1)})
        lime_df = lime_df.sort_values('value', ascending=False)
        lime_df.to_csv(lime_csv, index=False)

    # Agreement
    shap_rank = pd.read_csv(shap_csv).set_index('attr')
    lime_rank = pd.read_csv(lime_csv).set_index('attr')
    merged = shap_rank.join(lime_rank, lsuffix='_shap', rsuffix='_lime', how='inner')

    if len(merged) >= 2:
        corr, p = spearmanr(merged['value_shap'], merged['value_lime'])
    else:
        corr, p = np.nan, np.nan

    pd.DataFrame([{
        'track': track,
        'model': stack,
        'window': window,
        'spearman': corr,
        'p_value': p,
        'n_attrs': len(merged)
    }]).to_csv(corr_csv, index=False)

    print(f"[{track}] {stack} win{window}: SHAP/LIME done")


## 6) Integrated Gradients memory horizon + heatmap


In [ ]:
for track, stack, window, model_path in models_for_xai:
    assets = track_assets[track]
    base = assets['base']
    scaled_df = assets['scaled_df']

    ig_dir = base / 'analysis' / 'ig' / stack
    ig_dir.mkdir(parents=True, exist_ok=True)

    horizon_png = ig_dir / f'win{window}_horizon.png'
    heatmap_png = ig_dir / f'win{window}_heatmap.png'
    horizon_csv = ig_dir / f'win{window}_horizon.csv'

    if horizon_png.exists() and horizon_csv.exists():
        print(f"[{track}] {stack} win{window}: IG exists")
        continue

    model = load_model(str(model_path), custom_objects={'rmse': rmse})
    feature_cols = assets['feature_cols']
    X_train, X_test, y_train, y_test = build_window_data(scaled_df, feature_cols, window)

    baseline = X_train.mean(axis=0)
    curves = []

    n = min(IG_SAMPLES, len(X_test))
    for i in range(n):
        attr = integrated_gradients(model, X_test[i], baseline, steps=IG_STEPS)
        curves.append(np.abs(attr).mean(axis=1))

    horizon = np.mean(np.array(curves), axis=0)

    pd.DataFrame({'lag': np.arange(window), 'mean_abs_attr': horizon}).to_csv(horizon_csv, index=False)

    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(window), horizon, marker='o')
    plt.title(f'{track} {stack} win{window} memory horizon')
    plt.xlabel('Lag (steps back)')
    plt.ylabel('Mean |attr|')
    plt.tight_layout()
    plt.savefig(horizon_png, dpi=120)
    plt.close()

    # one sample heatmap
    heat = np.abs(integrated_gradients(model, X_test[0], baseline, steps=IG_STEPS))
    plt.figure(figsize=(10, 4))
    sns.heatmap(heat.T, cmap='magma')
    plt.title(f'{track} {stack} win{window} IG heatmap (sample 0)')
    plt.xlabel('Time step in window')
    plt.ylabel('Feature index')
    plt.tight_layout()
    plt.savefig(heatmap_png, dpi=120)
    plt.close()

    print(f"[{track}] {stack} win{window}: IG saved")


## 7) Memory erasure test (Gemini Step 2A)


In [ ]:
for track, stack, window, model_path in models_for_xai:
    assets = track_assets[track]
    base = assets['base']
    scaled_df = assets['scaled_df']
    scaler = assets['scaler']
    n_features = len(assets['feature_cols'])

    out_csv = base / 'analysis' / f'memory_erasure_{stack}_win{window}.csv'
    out_png = base / 'analysis' / f'memory_erasure_{stack}_win{window}.png'

    if out_csv.exists() and out_png.exists():
        print(f"[{track}] {stack} win{window}: memory erasure exists")
        continue

    model = load_model(str(model_path), custom_objects={'rmse': rmse})
    X_train, X_test, y_train, y_test = build_window_data(scaled_df, assets['feature_cols'], window)

    baseline_window = X_train.mean(axis=0)
    rows = []

    for cutoff in MEMORY_ERASURE_CUTOFFS:
        X_mod = X_test.copy()
        cut_idx = int(window * cutoff)
        if cut_idx > 0:
            X_mod[:, :cut_idx, :] = baseline_window[:cut_idx, :]

        pred = model.predict(X_mod, verbose=0).ravel()
        yt = to_kwh(y_test, scaler, 0, n_features)
        yp = to_kwh(pred, scaler, 0, n_features)
        m = compute_metrics(yt, yp)

        rows.append({'cutoff': cutoff, 'rmse_kwh': m['rmse'], 'mae_kwh': m['mae'], 'r2': m['r2'], 'wia': m['wia']})

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)

    plt.figure(figsize=(6, 4))
    plt.plot(df['cutoff'], df['rmse_kwh'], marker='o')
    plt.title(f'{track} {stack} win{window} memory erasure')
    plt.xlabel('Erased oldest fraction')
    plt.ylabel('RMSE (kWh)')
    plt.tight_layout()
    plt.savefig(out_png, dpi=120)
    plt.close()

    print(f"[{track}] {stack} win{window}: memory erasure saved")


## 8) Fidelity perturbation (Gemini Step 2B)


In [ ]:
for track, stack, window, model_path in models_for_xai:
    assets = track_assets[track]
    base = assets['base']
    scaled_df = assets['scaled_df']
    scaler = assets['scaler']
    feature_cols = assets['feature_cols']
    n_features = len(feature_cols)

    shap_csv = base / 'analysis' / 'xai' / stack / f'win{window}_shap.csv'
    lime_csv = base / 'analysis' / 'xai' / stack / f'win{window}_lime.csv'
    out_csv = base / 'analysis' / f'fidelity_{stack}_win{window}.csv'

    if out_csv.exists():
        print(f"[{track}] {stack} win{window}: fidelity exists")
        continue

    # pick top attributes from SHAP, else LIME
    top_attrs = []
    if shap_csv.exists():
        sdf = pd.read_csv(shap_csv)
        top_attrs = sdf['attr'].head(FIDELITY_TOP_K).tolist()
    elif lime_csv.exists():
        ldf = pd.read_csv(lime_csv)
        top_attrs = ldf['attr'].head(FIDELITY_TOP_K).tolist()

    if not top_attrs:
        print(f"[{track}] {stack} win{window}: no SHAP/LIME yet, skip fidelity")
        continue

    model = load_model(str(model_path), custom_objects={'rmse': rmse})
    X_train, X_test, y_train, y_test = build_window_data(scaled_df, feature_cols, window)

    # baseline
    base_pred = model.predict(X_test, verbose=0).ravel()
    yt = to_kwh(y_test, scaler, 0, n_features)
    yp_base = to_kwh(base_pred, scaler, 0, n_features)
    base_rmse = compute_metrics(yt, yp_base)['rmse']

    rows = []
    for attr in top_attrs:
        if attr not in feature_cols:
            continue
        idx = feature_cols.index(attr)

        # zero perturbation
        X_zero = X_test.copy()
        X_zero[:, :, idx] = 0.0
        yp_zero = to_kwh(model.predict(X_zero, verbose=0).ravel(), scaler, 0, n_features)
        rmse_zero = compute_metrics(yt, yp_zero)['rmse']

        # noise perturbation
        X_noise = X_test.copy()
        noise = np.random.normal(0, 0.1, size=X_noise[:, :, idx].shape)
        X_noise[:, :, idx] = X_noise[:, :, idx] + noise
        yp_noise = to_kwh(model.predict(X_noise, verbose=0).ravel(), scaler, 0, n_features)
        rmse_noise = compute_metrics(yt, yp_noise)['rmse']

        rows.append({
            'attr': attr,
            'baseline_rmse': base_rmse,
            'rmse_zero': rmse_zero,
            'rmse_noise': rmse_noise,
            'delta_zero': rmse_zero - base_rmse,
            'delta_noise': rmse_noise - base_rmse,
        })

    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[{track}] {stack} win{window}: fidelity saved")


## 9) Hidden-state brain map (PCA + t-SNE) for best model per track


In [ ]:
for track, best in best_models.items():
    assets = track_assets[track]
    base = assets['base']
    stack = best['model']
    window = best['window']

    model, model_path = load_saved_model(base, stack, window)
    if model is None:
        print(f"[{track}] best model missing, skip hidden-state")
        continue

    hidden_dir = base / 'analysis' / 'hidden_states'
    hidden_dir.mkdir(parents=True, exist_ok=True)

    pca_png = hidden_dir / f'pca_{stack}_win{window}.png'
    tsne_png = hidden_dir / f'tsne_{stack}_win{window}.png'

    X_train, X_test, y_train, y_test = build_window_data(assets['scaled_df'], assets['feature_cols'], window)

    # try first recurrent layer output
    layer_idx = 0
    layer = model.layers[layer_idx]
    submodel = tf.keras.Model(model.input, layer.output)

    sample_n = min(500, len(X_train))
    hs = submodel.predict(X_train[:sample_n], verbose=0)
    if hs.ndim == 3:
        hs = hs[:, -1, :]

    # PCA
    pca = PCA(n_components=2)
    coords = pca.fit_transform(hs)
    plt.figure(figsize=(6, 5))
    plt.scatter(coords[:, 0], coords[:, 1], s=10, alpha=0.6)
    plt.title(f'{track} hidden states PCA ({stack} win{window})')
    plt.tight_layout()
    plt.savefig(pca_png, dpi=120)
    plt.close()

    # t-SNE (can be slow)
    ts = TSNE(n_components=2, init='pca', learning_rate='auto', perplexity=30)
    tcoords = ts.fit_transform(hs)
    plt.figure(figsize=(6, 5))
    plt.scatter(tcoords[:, 0], tcoords[:, 1], s=10, alpha=0.6)
    plt.title(f'{track} hidden states t-SNE ({stack} win{window})')
    plt.tight_layout()
    plt.savefig(tsne_png, dpi=120)
    plt.close()

    print(f"[{track}] hidden-state plots saved")


## 10) Build per-track analysis report


In [ ]:
for track, assets in track_assets.items():
    base = assets['base']
    analysis_dir = base / 'analysis'
    analysis_dir.mkdir(parents=True, exist_ok=True)

    metrics_csv = analysis_dir / 'recomputed_metrics.csv'
    metrics_df = pd.read_csv(metrics_csv) if metrics_csv.exists() else pd.DataFrame()

    best = None
    if len(metrics_df) > 0:
        best_row = metrics_df.loc[metrics_df['rmse_kwh'].idxmin()]
        best = {'model': best_row['model'], 'window': int(best_row['window']), 'rmse_kwh': float(best_row['rmse_kwh'])}
    elif track in best_models:
        best = {'model': best_models[track]['model'], 'window': int(best_models[track]['window']), 'rmse_kwh': float(best_models[track]['rmse'])}

    lines = []
    lines.append(f"# Analysis report - {track}")
    lines.append("")
    lines.append("## Best model")
    if best:
        lines.append(f"- model: {best['model']}")
        lines.append(f"- window: {best['window']}")
        lines.append(f"- rmse_kwh: {best['rmse_kwh']:.4f}")
    else:
        lines.append("- best model not found")

    lines.append("")
    lines.append("## Top recomputed metrics")
    if len(metrics_df) > 0:
        lines.append(metrics_df.sort_values('rmse_kwh').head(10).to_string(index=False))
    else:
        lines.append("No recomputed_metrics.csv")

    lines.append("")
    lines.append("## SHAP/LIME/IG summary")
    lines.append("See analysis/xai/, analysis/ig/, memory_erasure_*.csv, fidelity_*.csv, hidden_states/")

    report_path = analysis_dir / 'analysis_report.md'
    report_path.write_text('
'.join(lines), encoding='utf-8')
    print(f"[{track}] wrote", report_path)
